# Milestone 2 Lab, notebook 4 of 6: Which model? Accuracy, cost, and speed

## Where we are

| Notebook | What you do |
|---|---|
| 1. Where we are, and the business's records | Recall the question and the answer key, and explore the records profit is calculated from |
| 2. One calculation, two engines | Write a calculation by hand in pandas and in SQL, with test cases that judge both |
| 3. A model drafts a transformation | Ask a model to write the calculation, and let the tests judge it |
| **4. Which model? Accuracy, cost, and speed** (you are here) | Give the same tasks to several models and compare them |
| 5. An agent that tests and fixes its own code | Give a model the tests as a tool, and compare it with a single draft |
| 6. The pipeline, and the same question again | Run the calculations on each engine and ask the analyst the leader's question |

Model requests in this notebook: **about 85 (about 25 with TINY on)**. A full run of this lab costs well under a dollar.

## What this notebook is for

Which model should draft your transformations?

It is a real decision with real trade-offs. A data team might run thousands of these requests. The most capable model is the most reliable and the most expensive. A small open-weight model costs a fraction as much and answers faster, but is it good enough?

In this notebook you stop guessing. You give the same tasks to several models, in SQL and in pandas, and measure three things: **accuracy, cost, and speed**.

You can only do this because of what you built in notebook 2. The hand-worked cases turn "is this model good?" from an opinion into a number.

> This notebook makes many model requests. With the models below, a full run costs a few cents and takes a few minutes. Set `TINY = True` for a quicker, cheaper run on two tasks.

In [1]:
import pandas as pd
import plotly.express as px
from lab_helpers import connect, run_comparison, save_result, show_code, show_doc, summarise

from retail_profit_stack.transform.drafting import draft_and_score, make_task_prompt
from retail_profit_stack.transform.specs import TASKS
from retail_profit_stack.transform.testing import load_cases

# A shorter wait and fewer retries than usual, so one slow model cannot hold everything up
client, MODEL = connect(timeout=90, max_retries=1)

Connected to OpenRouter. Default model: anthropic/claude-haiku-5.5


## Step 1: The question

This comes straight from the business requirements.

In [2]:
show_doc("BRD.md", "3. The questions to answer")

> **From `docs/BRD.md`: 3. The questions to answer**

The leader's question from milestone 1 is unchanged and must get the same answer:

> Which products and categories generated the most contribution profit last month, what changed, and how confident should I be in those numbers?

This milestone adds three questions of its own:

1. **For finance:** how exactly is each profit figure calculated from our records?
2. **For the head of data:** if we change the technology that runs the calculations, do we get the same figures?
3. **For the learner:** which model drafts a correct calculation most reliably, and what does it cost in money and time?

## Step 2: The tasks

Seven drafting tasks: the six transformations, and the all-in-one task from notebook 3. Each has its hand-worked cases. Together they are the exam.

In [3]:
TINY = False

tasks = [TASKS["product_cost"], TASKS["profit_all_in_one"]] if TINY else list(TASKS.values())

pd.DataFrame([{"task": task.name, "inputs": len(task.inputs), "test cases": len(load_cases(task.name))} for task in tasks])

,task,inputs,test cases
0,sales,2,5
1,refund_totals,1,3
2,delivery_costs,1,2
3,fee_totals,1,2
4,product_cost,2,3
5,monthly_product_profit,5,4
6,profit_all_in_one,6,3


## Step 3: The models

All of these are reached through OpenRouter with the same key and the same code. Nothing is installed on your computer.

> **Reminder: open-weight models.** Some models are closed: only their maker can run them. Others are **open-weight**: the model itself is published, so many companies can host it, often at lower prices. The list below has both kinds, across a range of sizes and prices.

Change this list to try other models. Each must support structured output.

In [4]:
MODELS = [
    "anthropic/claude-haiku-5.5",
    "deepseek/deepseek-v4.1-flash",
    "openai/gpt-oss-20b",
    "google/gemma-4-31b-it",
    "mistralai/mistral-small-2603",
    "meta-llama/llama-3.1-8b-instruct",
]

> **Reminder: tokens and cost.** Models read and write **tokens**, which are pieces of words. Providers charge per million tokens, with one price for what you send and a higher price for what the model writes. OpenRouter reports the cost of each request, so we simply add them up.

## Step 4: One draft, measured

`draft_and_score` does everything from notebook 3 in one call: build the prompt, ask the model, run the draft against the cases, and record how it did, along with the tokens, the cost, and the time.

In [5]:
first = draft_and_score(client, MODELS[0], TASKS["product_cost"], "sql")

print(f"{first.model} drafting {first.transformation} in {first.language}")
print(f"  cases passed:  {first.cases_passed} of {first.cases_total}")
print(f"  tokens:        {first.input_tokens} in, {first.output_tokens} out")
print(f"  cost:          ${first.cost:.5f}")
print(f"  time:          {first.seconds} seconds")

anthropic/claude-haiku-5.5 drafting product_cost in sql
  cases passed:  3 of 3
  tokens:        747 in, 664 out
  cost:          $0.00041
  time:          5.1 seconds


## Step 5: What a smaller prompt saves

In notebook 3 the prompt described only the tables the transformation reads. It would be easy to be lazy and describe every table in the system every time, "to give the model context".

Here is the same task with that bigger prompt. The model is asked for the same thing; it just has more to read.

In [6]:
trimmed = make_task_prompt(TASKS["product_cost"], "sql", "trimmed")
full = make_task_prompt(TASKS["product_cost"], "sql", "full")
print(f"Trimmed prompt: {len(trimmed.split()):,} words.  Full prompt: {len(full.split()):,} words.")

with_full = draft_and_score(client, MODELS[0], TASKS["product_cost"], "sql", context="full")

pd.DataFrame([
    {"prompt": "trimmed", "input tokens": first.input_tokens, "cost ($)": round(first.cost, 5), "cases passed": f"{first.cases_passed} of {first.cases_total}"},
    {"prompt": "full", "input tokens": with_full.input_tokens, "cost ($)": round(with_full.cost, 5), "cases passed": f"{with_full.cases_passed} of {with_full.cases_total}"},
])

Trimmed prompt: 211 words.  Full prompt: 509 words.


,prompt,input tokens,cost ($),cases passed
0,trimmed,747,0.00041,3 of 3
1,full,1705,0.00047,3 of 3


The full prompt costs more on every request and is rarely more accurate. A smaller prompt is cheaper and faster. Its risk is leaving out something the model needs, which is why you measure and don't assume.

We use the trimmed prompt from here on.

## Step 6: Run the comparison

Every model, every task, in both languages. The requests are sent several at a time, so this takes a few minutes, not half an hour. A model that fails to answer is scored as zero for that draft and the run carries on.

In [7]:
results = run_comparison(client, MODELS, tasks, ["sql", "pandas"])

84 drafts to request: 6 models x 7 tasks x 2 languages
  10 of 84 done after 10 seconds
  20 of 84 done after 21 seconds
  30 of 84 done after 36 seconds
  40 of 84 done after 50 seconds
  50 of 84 done after 63 seconds
  60 of 84 done after 69 seconds
  70 of 84 done after 77 seconds
  80 of 84 done after 159 seconds
  84 of 84 done after 181 seconds
Finished in 181 seconds. Total cost $0.0423


## Step 7: Accuracy, cost, and speed side by side

One row for each model.

- **cases passed (%)** is the share of all hand-worked cases its drafts passed.
- **fully correct** counts the drafts that passed *every* case. This is the number that matters most: a transformation that passes four cases out of five is still wrong.

In [8]:
summary = summarise(results)
summary

,model,cases passed (%),fully correct,cost per draft ($),seconds per draft
0,claude-haiku-5.5,100,14 of 14,0.00086,6.9
1,gpt-oss-20b,86,11 of 14,0.00024,10.7
2,gemma-4-31b-it,84,11 of 14,0.00024,7.7
3,mistral-small-2603,84,11 of 14,0.00035,3.4
4,deepseek-v4.1-flash,82,12 of 14,0.00122,13.3
5,llama-3.1-8b-instruct,7,1 of 14,0.00012,49.5


Does a model do better in one language than the other?

In [9]:
by_language = summarise(results, by=("model", "language"))
by_language.pivot(index="model", columns="language", values="cases passed (%)")

language,pandas,sql
model,,
claude-haiku-5.5,100,100
deepseek-v4.1-flash,86,77
gemma-4-31b-it,100,68
gpt-oss-20b,100,73
llama-3.1-8b-instruct,0,14
mistral-small-2603,82,86


And which tasks were hard? This counts, for each task, how many drafts were fully correct across all models and both languages.

In [10]:
frame = pd.DataFrame([result.model_dump() for result in results])
by_task = frame.groupby("transformation", sort=False)["all_passed"].agg(["sum", "size"])
by_task["fully correct"] = by_task["sum"].astype(str) + " of " + by_task["size"].astype(str)
by_task[["fully correct"]]

,fully correct
transformation,
sales,8 of 12
refund_totals,9 of 12
delivery_costs,10 of 12
fee_totals,8 of 12
product_cost,10 of 12
monthly_product_profit,10 of 12
profit_all_in_one,5 of 12


### Accuracy against cost

The best place to be is the top left: accurate and cheap.

In [11]:
fig = px.scatter(summary, x="cost per draft ($)", y="cases passed (%)", text="model", title="Accuracy against cost", height=520)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 105])
fig.show()

### Accuracy against speed

Again the top left is best: accurate and fast.

In [12]:
fig = px.scatter(summary, x="seconds per draft", y="cases passed (%)", text="model", title="Accuracy against speed", height=520)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 105])
fig.show()

## Step 8: Look at a failure

Numbers tell you which model struggled. To learn *why*, read a failing draft. This picks one draft that did not pass every case, and shows the code beside the first test it failed.

In [13]:
failures = [result for result in results if result.code and not result.all_passed]

if failures:
    worst = failures[0]
    print(f"{worst.model} drafting {worst.transformation} in {worst.language}: {worst.cases_passed} of {worst.cases_total} cases passed")
    print("First failure:", worst.failure)
    show_code(worst.code, worst.language)
else:
    print("Every draft that came back passed every case this time.")

deepseek/deepseek-v4.1-flash drafting sales in sql: 0 of 5 cases passed
First failure: Not run: The SQL could not be read: Parser Error: syntax error at or near "dummy"

LINE 1: dummy
        ^^^^^


```sql
dummy
```

Common reasons you will see: joining order lines straight to refunds (the trap from notebook 2), dropping the product with no cost or filling in zero, the month as a date and not as text like `2026-10`, a column missing or misnamed, and forgetting cancelled orders.

**Before you blame the model, read your prompt.** When this lab was built, the first version of the prompt explained how to handle dates on every task, including tasks that have no dates in them. Three of the models applied that advice where it didn't belong and failed. Changing the prompt to mention dates only where a task has them lifted those models from between 64% and 86% of cases passed to 93% or more. Nothing about the models had changed.

> ### Commercial implications
>
> Weighing accuracy against cost and speed, for a specific business task, is one of the most useful things an AI data engineer does. The answer is rarely "the biggest model". If a small model is fully correct on your tasks at a tenth of the price, that is the one to use, and you can prove it.
>
> There is a second consideration this lab doesn't measure: where your data and your table designs are allowed to go. Open-weight models can be run inside a company's own systems. For some businesses that decides the question before price does.

## What we learned

- **You can choose a model on evidence.** The same tasks, the same tests, and three numbers per model.
- **Accuracy means "fully correct" here.** Nearly right is wrong when the output is a financial figure.
- **A smaller prompt saves money on every request.** Measure whether it costs you accuracy.
- **The prompt is part of the system.** A misleading sentence in it can cost more accuracy than the choice of model.
- **The comparison is only as good as the test cases.** They are the part a person has to get right.

Some drafts failed a case or two. A person would read the failure and fix the code. Could the model do that itself?

**Next: notebook 5**, where a model is given the tests as a tool.

In [14]:
save_result("model_comparison", {"summary": summary.to_dict("records"), "results": [result.model_dump() for result in results]})

Saved results/model_comparison.json
